# 02 — Data Quality Assessment

**Project:** Customer Segmentation Using Unsupervised Machine Learning  
**Dataset:** UCI Online Retail  
**Unit of analysis:** Transaction-level records (before customer aggregation)

## Purpose

This notebook assesses the quality and reliability of the original transaction dataset before customer-level feature engineering. It examines missing values, exact duplicates, cancellations, negative quantities, unusual prices and extreme transaction values.

The original dataset is preserved. All investigations and exclusions are documented so that subsequent preprocessing decisions are transparent and reproducible.

**Expected output:** A data-quality assessment that supports the cleaning rules used in Notebook 04.

## 2.1 Load the original dataset and prepare the analysis environment

The data is loaded through the project's shared loading function. The original transaction-level DataFrame is retained as `df`; subsequent investigations use derived copies where necessary.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_online_retail
from src.data_quality import missing_value_summary, core_quality_metrics

df = load_online_retail()

print(f"Original dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Original dataset shape: 541,909 rows × 8 columns


The original dataset contains **541,909 transaction records and 8 variables**. These records are the starting point for the audit; customer-level filtering and aggregation are performed later in the workflow.

## 2.2 Missing-value assessment

Missing values are quantified by variable using both counts and percentages. This is particularly important because CustomerID is required to assign transactions to individual customers.

In [2]:
missing = missing_value_summary(df)
display(missing)

missing.to_csv(
    PROJECT_ROOT / "reports" / "phase_2" / "tables" / "missing_values.csv",
    index=False
)

,variable,data_type,missing_count,missing_pct,unique_nonmissing
0,InvoiceNo,object,0,0.00,25900
1,StockCode,object,0,0.00,4070
2,Description,object,1454,0.27,4223
3,Quantity,int64,0,0.00,722
4,InvoiceDate,datetime64[us],0,0.00,23260
5,UnitPrice,float64,0,0.00,1630
6,CustomerID,float64,135080,24.93,4372
7,Country,str,0,0.00,38


### Interpretation of missing-value assessment

The missing-value assessment indicates that CustomerID has the highest proportion of missing observations, with 135,080 records (24.93%) lacking a customer identifier. This is a significant limitation for customer-level analysis because customer identification is necessary to construct Recency, Frequency and Monetary features.

Description has 1,454 missing observations (0.27%). Since StockCode is available and can be used to identify products, missing product descriptions are not expected to materially affect the primary segmentation analysis.

The remaining six variables have no missing observations.

**Preliminary preprocessing decisions:**

1. Transactions with missing CustomerID will be excluded from customer-level aggregation and clustering, but retained in the original dataset and data-quality audit.
2. Missing CustomerID values will not be imputed because customer identity cannot be reliably inferred from the available information.
3. Records with missing Description will be retained, as StockCode provides an alternative product identifier.

These decisions will be revisited alongside the investigation of cancellations, returns and other unusual transaction records.

## 2.3 Duplicate records and dataset structure

Exact duplicates are identified across all eight original variables. We distinguish the number of rows involved in duplicate groups from the number of excess rows that would be removed by `drop_duplicates()`.

We also inspect whether duplicate rows occur in purchase and cancellation records, because their treatment may affect transaction-level matching.

In [3]:
metrics = core_quality_metrics(df)
display(metrics.to_frame())

metrics.to_csv(
    PROJECT_ROOT / "reports" / "phase_2" / "tables" / "core_quality_metrics.csv"
)

print(f"Exact duplicate excess rows: {df.duplicated().sum():,}")

,value
observations,541909
variables,8
exact_duplicate_excess_rows,5268
unique_customers_nonmissing,4372
unique_invoices,25900
unique_products,4070
countries,38
c_prefixed_rows,9288
distinct_c_prefixed_invoices,3836
negative_quantity_rows,10624


Exact duplicate excess rows: 5,268


### Interpretation of duplicate records

The initial data-quality assessment identified 5,268 excess exact duplicate rows among the 541,909 original transaction records, representing approximately 0.97% of the dataset.

These records have identical values across all eight variables. However, identical transaction records cannot automatically be assumed to be erroneous because the available dataset does not provide a unique transaction-line identifier to distinguish accidental duplication from legitimate repeated entries.

For the primary analytical dataset, exact duplicate rows will be removed while preserving the original dataset. A sensitivity analysis will subsequently compare the results with a version retaining these records to assess whether duplicate handling materially influences customer segmentation.

In [4]:
# Identify cancellation records without modifying the original DataFrame.
invoice_text = df["InvoiceNo"].astype("string").str.strip()
df_audit = df.copy()
df_audit["Is_Cancellation"] = invoice_text.str.upper().str.startswith("C", na=False)

duplicate_rows = df_audit[
    df_audit.duplicated(subset=df.columns.tolist(), keep=False)
].copy()

duplicate_excess = df_audit[
    df_audit.duplicated(subset=df.columns.tolist(), keep="first")
].copy()

print("DUPLICATE GROUP INVESTIGATION")
print("-" * 45)
print(f"Rows involved in duplicate groups: {len(duplicate_rows):,}")
print(f"Excess duplicate rows: {len(duplicate_excess):,}")

print("\nRows involved in duplicate groups by transaction type:")
display(
    duplicate_rows["Is_Cancellation"]
    .value_counts()
    .rename(index={False: "Non-cancellation", True: "Cancellation"})
    .to_frame("Rows involved")
)

print("\nExcess duplicate rows by transaction type:")
display(
    duplicate_excess["Is_Cancellation"]
    .value_counts()
    .rename(index={False: "Non-cancellation", True: "Cancellation"})
    .to_frame("Excess duplicate rows")
)

DUPLICATE GROUP INVESTIGATION
---------------------------------------------
Rows involved in duplicate groups: 10,147
Excess duplicate rows: 5,268

Rows involved in duplicate groups by transaction type:


,Rows involved
Is_Cancellation,
Non-cancellation,10078
Cancellation,69



Excess duplicate rows by transaction type:


,Excess duplicate rows
Is_Cancellation,
Non-cancellation,5231
Cancellation,37


The duplicate investigation separates two questions: how many rows belong to duplicate groups, and how many excess copies would be removed. The latter is the count relevant to deduplication.

Exact duplicates are removed from the primary analytical copy in the preprocessing stage, while the original data remains unchanged. The transaction-level audit below also uses deduplicated temporary copies so repeated rows do not artificially multiply candidate purchase–cancellation matches.

## 2.4 Cancellation and purchase-reversal investigation

C-prefixed invoice numbers identify cancellation records. A cancellation cannot automatically be linked to a specific earlier purchase using invoice number alone, so a second exploratory check compares customer, product, absolute quantity, unit price and transaction chronology.

The matching exercise is diagnostic only. Candidate matches are not treated as confirmed reversals where more than one earlier purchase could satisfy the matching conditions.

In [5]:
# Separate purchase-side and cancellation records.
purchase_records = df_audit.loc[~df_audit["Is_Cancellation"]].dropna(
    subset=["CustomerID"]
).copy()

cancellation_records = df_audit.loc[df_audit["Is_Cancellation"]].dropna(
    subset=["CustomerID"]
).copy()

# Deduplicate temporary copies for matching only; original df is unchanged.
purchase_clean = (
    purchase_records
    .drop_duplicates(subset=df.columns.tolist())
    .reset_index(drop=True)
)
cancellation_clean = (
    cancellation_records
    .drop_duplicates(subset=df.columns.tolist())
    .reset_index(drop=True)
)

# Add row identifiers and make cancellation quantity positive for matching.
purchase_clean["Purchase_Row_ID"] = purchase_clean.index
cancellation_clean["Cancellation_Row_ID"] = cancellation_clean.index
cancellation_clean["Abs_Quantity"] = cancellation_clean["Quantity"].abs()

# Candidate matches require the same customer, product, absolute quantity
# and unit price, with the purchase occurring no later than the cancellation.
matches_clean = cancellation_clean.merge(
    purchase_clean,
    left_on=["CustomerID", "StockCode", "Abs_Quantity", "UnitPrice"],
    right_on=["CustomerID", "StockCode", "Quantity", "UnitPrice"],
    suffixes=("_cancel", "_purchase")
)

matches_clean = matches_clean.loc[
    matches_clean["InvoiceDate_purchase"] <= matches_clean["InvoiceDate_cancel"]
].copy()

candidate_counts = (
    matches_clean.groupby("Cancellation_Row_ID")["Purchase_Row_ID"].nunique()
)

same_timestamp = (
    matches_clean["InvoiceDate_purchase"]
    == matches_clean["InvoiceDate_cancel"]
)

print("DUPLICATE-AWARE PURCHASE–CANCELLATION MATCHING")
print("-" * 55)
print(f"Unique cancellation records with CustomerID: {len(cancellation_clean):,}")
print(f"Unique purchase records with CustomerID: {len(purchase_clean):,}")
print(f"Cancellation records with at least one candidate: {candidate_counts.size:,}")
print(f"Cancellation records with exactly one candidate: {(candidate_counts == 1).sum():,}")
print(f"Cancellation records with multiple candidates: {(candidate_counts > 1).sum():,}")
print(f"Unique purchase records in candidate matches: {matches_clean['Purchase_Row_ID'].nunique():,}")
print(f"Total candidate matching pairs: {len(matches_clean):,}")
print(f"Candidate pairs with identical timestamps: {same_timestamp.sum():,}")

display(
    matches_clean[
        [
            "InvoiceNo_purchase", "InvoiceNo_cancel", "CustomerID",
            "StockCode", "Quantity_purchase", "Quantity_cancel",
            "InvoiceDate_purchase", "InvoiceDate_cancel", "UnitPrice"
        ]
    ].head(15)
)

DUPLICATE-AWARE PURCHASE–CANCELLATION MATCHING
-------------------------------------------------------
Unique cancellation records with CustomerID: 8,872
Unique purchase records with CustomerID: 392,732
Cancellation records with at least one candidate: 2,798
Cancellation records with exactly one candidate: 2,222
Cancellation records with multiple candidates: 576
Unique purchase records in candidate matches: 3,830
Total candidate matching pairs: 3,953
Candidate pairs with identical timestamps: 8


,InvoiceNo_purchase,InvoiceNo_cancel,CustomerID,StockCode,Quantity_purchase,Quantity_cancel,InvoiceDate_purchase,InvoiceDate_cancel,UnitPrice
15,536557,C536979,17841.0,84685,1,-1,2010-12-01 14:41:00,2010-12-03 14:23:00,3.75
22,537140,C537143,12748.0,20685,1,-1,2010-12-05 12:53:00,2010-12-05 12:58:00,7.95
24,537144,C537157,15880.0,35953,24,-24,2010-12-05 13:00:00,2010-12-05 13:09:00,1.25
25,536591,C537203,14606.0,37449,1,-1,2010-12-01 16:57:00,2010-12-05 14:44:00,9.95
28,537298,C537314,13497.0,21349,4,-4,2010-12-06 11:37:00,2010-12-06 11:42:00,6.75
32,537377,C537379,15882.0,22865,10,-10,2010-12-06 13:01:00,2010-12-06 13:12:00,2.10
34,537196,C537398,15426.0,22430,1,-1,2010-12-05 13:55:00,2010-12-06 14:22:00,4.95
35,537217,C537402,15502.0,22926,4,-4,2010-12-05 15:40:00,2010-12-06 14:43:00,5.95
37,537217,C537402,15502.0,22927,4,-4,2010-12-05 15:40:00,2010-12-06 14:43:00,5.95
39,537217,C537402,15502.0,22849,4,-4,2010-12-05 15:40:00,2010-12-06 14:43:00,14.95


In [6]:
# Summarise candidate matches at cancellation-invoice level.
invoice_match_summary = (
    matches_clean
    .groupby("InvoiceNo_cancel")
    .agg(
        candidate_cancellation_lines=("Cancellation_Row_ID", "nunique"),
        candidate_purchase_lines=("Purchase_Row_ID", "nunique"),
        customers=("CustomerID", "nunique")
    )
    .reset_index()
)

print(
    "Distinct cancellation invoices with candidate matches:",
    invoice_match_summary["InvoiceNo_cancel"].nunique()
)

print("\nCandidate purchase lines per cancellation invoice:")
display(
    invoice_match_summary["candidate_purchase_lines"]
    .value_counts()
    .sort_index()
    .rename_axis("Candidate purchase lines")
    .to_frame("Cancellation invoices")
)

Distinct cancellation invoices with candidate matches: 1237

Candidate purchase lines per cancellation invoice:


,Cancellation invoices
Candidate purchase lines,
1,734
2,215
3,74
4,54
5,23
6,19
7,14
8,15
9,11


### Interpretation: cancellation matching

After removing exact duplicates from temporary matching copies, **8,872 cancellation records with non-missing CustomerIDs** and **392,732 purchase-side records** were available for investigation.

The matching procedure identified **2,798 cancellation records** with at least one candidate purchase. Of these, **2,222 had exactly one candidate** and **576 had multiple candidates**. The candidate matches involved **3,830 unique purchase records** across **1,237 cancellation invoices**.

These are potential relationships, not confirmed reversals. A customer may purchase the same product in multiple invoices, and the dataset does not provide a unique transaction-line identifier linking a cancellation to its original purchase. Therefore, the analysis does not subtract candidate cancellations from customer spending.

**Decision:** C-prefixed cancellation invoices are excluded from the primary purchase-based RFM calculations and retained separately. Exact purchase–cancellation matching is not used to adjust Monetary values.

## 2.5 Negative quantities and unusual prices

This section quantifies cancellation records, negative quantities without a C prefix, and zero or negative unit prices. These categories are investigated separately because a negative quantity is not necessarily equivalent to a conventional cancellation.

In [7]:
invoice_text = df["InvoiceNo"].astype("string")
is_cancel = invoice_text.str.startswith("C", na=False)
non_c_negative = (df["Quantity"] < 0) & (~is_cancel)

exception_summary = pd.Series({
    "C-prefixed rows": int(is_cancel.sum()),
    "Distinct C-prefixed invoices": int(df.loc[is_cancel, "InvoiceNo"].nunique()),
    "Negative Quantity rows": int((df["Quantity"] < 0).sum()),
    "Negative Quantity without C prefix": int(non_c_negative.sum()),
    "Zero UnitPrice": int((df["UnitPrice"] == 0).sum()),
    "Negative UnitPrice": int((df["UnitPrice"] < 0).sum()),
    "Non-C negative rows all CustomerID missing": bool(
        df.loc[non_c_negative, "CustomerID"].isna().all()
    ),
    "Non-C negative rows all UnitPrice zero": bool(
        (df.loc[non_c_negative, "UnitPrice"] == 0).all()
    ),
})

display(exception_summary.to_frame("Value"))

exception_summary.to_csv(
    PROJECT_ROOT / "reports" / "phase_2" / "tables" / "cancellation_return_checks.csv"
)

,Value
C-prefixed rows,9288
Distinct C-prefixed invoices,3836
Negative Quantity rows,10624
Negative Quantity without C prefix,1336
Zero UnitPrice,2515
Negative UnitPrice,2
Non-C negative rows all CustomerID missing,True
Non-C negative rows all UnitPrice zero,True


The audit identifies **9,288 C-prefixed rows across 3,836 cancellation invoices** and **10,624 negative-quantity rows overall**. The 1,336 negative-quantity rows without a C prefix all have missing CustomerIDs and zero UnitPrice. They therefore cannot be reliably assigned to identifiable customers and will not contribute to customer-level purchase features.

### Zero-unit-price records

Zero-price records are examined separately because they may represent complimentary items, operational adjustments or other non-revenue activity. They should not automatically be interpreted as erroneous purchases.

In [8]:
# Identify zero-price records
zero_price_records = df[
    df["UnitPrice"] == 0
].copy()

print("ZERO-PRICE RECORD INVESTIGATION")
print("-" * 45)

print("Total zero-price records:",
      len(zero_price_records))

print("Records with CustomerID:",
      zero_price_records["CustomerID"].notna().sum())

print("Records without CustomerID:",
      zero_price_records["CustomerID"].isna().sum())

print("\nQuantity distribution:")

display(
    zero_price_records["Quantity"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nMost frequent zero-price product descriptions:")

display(
    zero_price_records["Description"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nSample zero-price records:")

display(
    zero_price_records[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "CustomerID",
            "Country"
        ]
    ].head(20)
)

ZERO-PRICE RECORD INVESTIGATION
---------------------------------------------
Total zero-price records: 2515
Records with CustomerID: 40
Records without CustomerID: 2475

Quantity distribution:


Quantity
-9600    2
-9058    1
-5368    1
-4830    1
-3667    1
-3167    1
-3100    1
-3000    1
-2880    1
-2834    1
-2618    1
-2600    1
-2472    1
-2376    1
-2003    1
-1897    1
-1681    1
-1671    1
-1560    1
-1512    1
Name: count, dtype: int64


Most frequent zero-price product descriptions:


Description
NaN                              1454
check                             159
?                                  47
damages                            45
damaged                            43
found                              25
sold as set on dotcom              20
adjustment                         16
Damaged                            14
FRENCH BLUE METAL DOOR SIGN 1       9
thrown away                         9
Unsaleable, destroyed.              9
amazon                              8
FRENCH BLUE METAL DOOR SIGN 8       8
Found                               8
Name: count, dtype: int64


Sample zero-price records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,NaN,United Kingdom


### Interpretation of zero-unit-price records

The dataset contains 2,515 records with UnitPrice equal to zero. Of these, 2,475 (98.41%) have missing CustomerIDs, while only 40 records are associated with identifiable customers.

The description analysis indicates that zero-price records frequently contain entries such as "check", "damages", "damaged", "found", "adjustment", "thrown away" and "Unsaleable, destroyed." These descriptions suggest that a substantial proportion of such records may represent operational or inventory-related adjustments rather than conventional customer purchases.

The sample also contains negative-quantity records with zero unit prices, further indicating that zero-price observations should not automatically be interpreted as complimentary purchases.

**Preliminary preprocessing decisions:**

1. Zero-price records without CustomerIDs will be excluded from customer-level analysis.
2. Zero-price records associated with identifiable customers will be examined individually before determining their treatment.
3. Zero-price records will not contribute to Monetary calculations.
4. Operational adjustments will not automatically contribute to customer quantity or product-diversity features.
5. All zero-price records will be retained in the original dataset and documented in the data-quality assessment.

In [9]:
# Isolate zero-price records associated with identified customers

zero_price_identified = zero_price_records[
    zero_price_records["CustomerID"].notna()
].copy()

zero_price_identified["Is_Cancellation"] = (
    zero_price_identified["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

print("IDENTIFIABLE ZERO-PRICE RECORDS")
print("-" * 45)

print("Total records:", len(zero_price_identified))

print("Unique customers:",
      zero_price_identified["CustomerID"].nunique())

print("\nQuantity sign distribution:")

display(
    zero_price_identified["Quantity"]
    .apply(
        lambda x: "Positive" if x > 0
        else "Negative" if x < 0
        else "Zero"
    )
    .value_counts()
    .to_frame("Records")
)

print("\nCancellation status:")

display(
    zero_price_identified["Is_Cancellation"]
    .value_counts()
    .rename(index={True: "Cancellation", False: "Non-cancellation"})
    .to_frame("Records")
)

print("\nDescription distribution:")

display(
    zero_price_identified["Description"]
    .value_counts(dropna=False)
    .to_frame("Records")
)

print("\nDetailed records:")

display(
    zero_price_identified.sort_values(
        ["CustomerID", "InvoiceDate"]
    )[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "CustomerID",
            "Country"
        ]
    ]
)

IDENTIFIABLE ZERO-PRICE RECORDS
---------------------------------------------
Total records: 40
Unique customers: 31

Quantity sign distribution:


,Records
Quantity,
Positive,40



Cancellation status:


,Records
Is_Cancellation,
Non-cancellation,40



Description distribution:


,Records
Description,
Manual,6
ROUND CAKE TIN VINTAGE GREEN,1
ADVENT CALENDAR GINGHAM SACK,1
REGENCY CAKESTAND 3 TIER,1
PAPER BUNTING RETROSPOT,1
PLASTERS IN TIN SKULLS,1
ORGANISER WOOD ANTIQUE WHITE,1
FAIRY CAKES NOTEBOOK A6 SIZE,1
CERAMIC BOWL WITH LOVE HEART DESIGN,1



Detailed records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,CustomerID,Country
198383,554037,22619,SET OF 6 SOLDIER SKITTLES,80,2011-05-20 14:13:00,12415.0,Australia
436428,574138,23234,BISCUIT TIN VINTAGE CHRISTMAS,216,2011-11-03 11:26:00,12415.0,Australia
439361,574469,22385,JUMBO BAG SPACEBOY DESIGN,12,2011-11-04 11:55:00,12431.0,Australia
436961,574252,M,Manual,1,2011-11-03 13:24:00,12437.0,France
480649,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,2011-11-18 13:23:00,12444.0,Norway
395529,571035,M,Manual,1,2011-10-13 12:50:00,12446.0,RSA
157042,550188,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,2011-04-14 18:57:00,12457.0,Switzerland
282912,561669,22960,JAM MAKING SET WITH JARS,11,2011-07-28 17:09:00,12507.0,Spain
479546,577168,M,Manual,1,2011-11-18 10:42:00,12603.0,Germany
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,12647.0,Germany


### Final interpretation of zero-price records

The investigation identified 2,515 records with zero UnitPrice. Of these, 2,475 (98.41%) had missing CustomerIDs, while 40 records were associated with identifiable customers.

The 40 identifiable records represented 31 unique customers, 34 distinct invoices and 35 distinct products. All 40 records had positive quantities, non-C-prefixed invoice numbers and no exact duplicates.

Although several records contained descriptions corresponding to ordinary retail products, others included entries such as "Manual" and "PADS TO MATCH ALL CUSHIONS". An unusually large quantity of 12,540 units was also observed for one zero-price record.

The available information does not establish whether these records represent complimentary products, promotional transactions or operational adjustments.

**Final preprocessing decision:** All zero-price records will be excluded from the primary paid-purchase analytical dataset. They will not contribute to Monetary, paid-purchase Frequency, Quantity or Product Diversity. The records will be retained separately for supplementary investigation and potential sensitivity analysis.

### Negative unit-price records

In [10]:
# Investigate negative unit-price records

negative_price_records = df[
    df["UnitPrice"] < 0
].copy()

print("NEGATIVE UNIT-PRICE INVESTIGATION")
print("-" * 45)

print("Total records:", len(negative_price_records))

print("\nDetailed records:")

display(
    negative_price_records[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]
    ]
)

print("\nDescription distribution:")

display(
    negative_price_records["Description"]
    .value_counts(dropna=False)
)

print("\nInvoice number classification:")

display(
    negative_price_records["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .value_counts()
)

print("\nCustomerID missing status:")

print(
    negative_price_records["CustomerID"].isna().value_counts()
)

NEGATIVE UNIT-PRICE INVESTIGATION
---------------------------------------------
Total records: 2

Detailed records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom



Description distribution:


Description
Adjust bad debt    2
Name: count, dtype: int64


Invoice number classification:


InvoiceNo
False    2
Name: count, dtype: int64


CustomerID missing status:
CustomerID
True    2
Name: count, dtype: int64


### Interpretation of negative unit-price records

The dataset contains two observations with negative UnitPrice values. Both records have StockCode "B" and the description "Adjust bad debt", with a recorded UnitPrice of -11,062.06.

The observations were recorded on 12 August 2011 under invoice identifiers A563186 and A563187. Both have missing CustomerIDs and positive quantities of one unit.

Based on their descriptions and transaction characteristics, these records represent accounting-related adjustments rather than conventional retail purchases.

**Preprocessing decision:** Both records will be excluded from the primary customer-level analytical dataset because they do not represent identifiable customer purchases. They will be retained in the original dataset and documented as accounting-related adjustments.

## 2.6 Distribution and extreme-value inspection

Descriptive percentiles and targeted record inspections are used to understand the scale and skewness of Quantity, UnitPrice and transaction line value. Extreme values are investigated in context rather than removed solely because they exceed a statistical threshold.

In [11]:
for column in ["Quantity", "UnitPrice"]:
    print(f"Distribution of {column}:")
    display(
        df[column]
        .describe(percentiles=[0.50, 0.90, 0.95, 0.99, 0.995, 0.999])
        .to_frame()
    )

# Audit-only transaction line value; this does not modify the source data.
line_value = df["Quantity"] * df["UnitPrice"]
display(
    line_value.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99, 0.995, 0.999]
    ).to_frame("Transaction line value")
)

# Inspect records with the largest absolute quantities and line values.
quantity_extremes = df.loc[
    df["Quantity"].abs().nlargest(15).index,
    [
        "InvoiceNo", "StockCode", "Description", "Quantity",
        "InvoiceDate", "UnitPrice", "CustomerID", "Country"
    ]
]

line_value_extremes = df.loc[
    line_value.abs().nlargest(15).index,
    [
        "InvoiceNo", "StockCode", "Description", "Quantity",
        "InvoiceDate", "UnitPrice", "CustomerID", "Country"
    ]
]

print("Records with the largest absolute quantities:")
display(quantity_extremes)

print("Records with the largest absolute transaction line values:")
display(line_value_extremes)

Distribution of Quantity:


,Quantity
count,541909.000000
mean,9.552250
std,218.081158
min,-80995.000000
50%,3.000000
90%,24.000000
95%,29.000000
99%,100.000000
99.5%,160.000000
99.9%,468.920000


Distribution of UnitPrice:


,UnitPrice
count,541909.000000
mean,4.611114
std,96.759853
min,-11062.060000
50%,2.080000
90%,7.950000
95%,9.950000
99%,18.000000
99.5%,24.960000
99.9%,205.559800


,Transaction line value
count,541909.000000
mean,17.987795
std,378.810824
min,-168469.600000
50%,9.750000
90%,31.800000
95%,59.400000
99%,180.000000
99.5%,288.000000
99.9%,825.000000


Records with the largest absolute quantities:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
225529,556690,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
225530,556691,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom
225528,556687,23003,Printing smudges/thrown away,-9058,2011-06-14 10:36:00,0.00,NaN,United Kingdom
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom


Records with the largest absolute transaction line values:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
222680,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom
222681,C556445,M,Manual,-1,2011-06-10 15:31:00,38970.00,15098.0,United Kingdom
524602,C580605,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:36:00,17836.46,NaN,United Kingdom
43702,C540117,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:55:00,16888.02,NaN,United Kingdom
43703,C540118,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:57:00,16453.71,NaN,United Kingdom
15016,C537630,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:04:00,13541.33,NaN,United Kingdom


### Interpretation of distribution and extreme values

The descriptive statistics reveal substantial dispersion and right-skewness in Quantity, UnitPrice and transaction line values.

Quantity has a median of 3 units and a mean of 9.55 units, with the 99th percentile at 100 units and the 99.9th percentile at approximately 469 units. The maximum and minimum values are both 80,995 in absolute terms.

UnitPrice has a median of £2.08 and a mean of £4.61, while its maximum reaches £38,970. Transaction line value, calculated as Quantity multiplied by UnitPrice, has a median of £9.75 and an absolute maximum of £168,469.60.

Inspection of extreme records identified two particularly large positive purchases, involving 80,995 units for CustomerID 16446 and 74,215 units for CustomerID 12346. Both have corresponding C-prefixed cancellation records with matching customer, product, quantity and price characteristics, recorded shortly after the original transactions.

Other large positive quantities were observed for identifiable customers at positive unit prices. These may represent genuine bulk purchases and should not automatically be removed as outliers.

The investigation also identified operational and non-standard entries, including zero-price records, manual adjustments and accounting-related transactions.

**Preprocessing decisions:**

1. C-prefixed cancellation records will be excluded from primary purchase-based RFM calculations.
2. Zero-price and negative-price records will be excluded from the primary paid-purchase dataset.
3. Non-standard operational and accounting entries will be investigated before finalising the analytical filter.
4. Legitimate high-volume and high-value purchases will be retained rather than automatically removed using percentile thresholds.
5. Extreme observations will be documented and their influence assessed during subsequent customer-level analysis.

## 2.7 Consolidated data-quality findings and preprocessing implications

The assessment identifies several issues that must be addressed before customer-level feature engineering.

| Issue | Finding | Implication for subsequent analysis |
|---|---|---|
| Missing CustomerID | 135,080 records (24.93%) | Exclude from customer-level aggregation; do not impute identities |
| Missing Description | 1,454 records (0.27%) | Retain where StockCode is available |
| Exact duplicates | 5,268 excess rows | Remove from the primary analytical copy; preserve raw data |
| C-prefixed cancellations | 9,288 rows across 3,836 invoices | Exclude from primary completed-purchase calculations |
| Non-C negative quantities | 1,336 rows; all lack CustomerID and have zero price | Exclude from customer-level purchase features |
| Zero UnitPrice | 2,515 rows; 40 have CustomerID | Exclude from primary paid-purchase features; retain for audit |
| Negative UnitPrice | 2 accounting adjustment rows | Exclude from customer-level purchase analysis |
| Extreme values | Very large quantities and transaction values | Investigate; do not automatically trim or cap legitimate bulk purchases |

### Overall conclusion

The original dataset contains sufficient identifiable purchase information for customer-level segmentation, but several transaction types cannot be treated as ordinary paid purchases. Missing customer identifiers, cancellations, zero-price records, accounting adjustments and exact duplicates require explicit treatment.

The audit supports a conservative preprocessing strategy: preserve the raw data, exclude records that cannot be reliably interpreted as identifiable paid purchases, and retain potentially legitimate high-volume transactions unless there is evidence that they are erroneous. The influence of the two unusually large, closely matched purchase–cancellation pairs is assessed later through sensitivity analysis.

**Transition to Notebook 03:** The next notebook explores the transaction and customer patterns in the original data. The cleaning rules established here will be applied systematically in Notebook 04 before the final customer-level modelling dataset is constructed.